# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
%pip install google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

import os
import json



## 1. Configurare — mai multe modele

In [18]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
    ("groq", "llama-3.1-8b-instant", "LLaMA 3 8B (Groq)"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free', 'LLaMA 3 8B (Groq)']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1",
    "groq": "https://api.groq.com/openai/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY"),
    "groq": os.getenv("GROQ_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("groq")

prompt = "Explică în 2 propoziții ce este un LLM."

response = client.chat.completions.create(
    model="llama-3.1-8b-instant", 
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)


# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content


# apel corect (Gemini)
raspuns = ask(
    provider="groq",
    model="llama-3.1-8b-instant",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un model de inteligență artificială care este capabil să înțeleagă și să genereze text, precum cărți, articoluri, conversații, etc., pe baza unei analize lingvistice și utilizării largi de date de învățare. LLM-urile sunt capabile să învețe și să se specializeze pe o varietate de domenii și să ofere răspunsuri personalizate prin intermediul unor algoritmi sofisticati.
Un LLM (Large Language Model) este un model de limbă artificial avansat care este capabil să proceseze și să genereze texte pe baza unui set de date mari și complexe. În general, LLM-urile sunt învățate pe baza unui corpus mare de texte și pot răspunde la întrebări, genera texte creative, traduce texte dintr-o limbă în alta, și oferă informații utile pe o gamă largă de subiecte.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError, APITimeoutError
import json


def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"
    
    except APITimeoutError:
        return "[Eroare: request timeout.]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Rezumă în exact 2 propoziții scurte, în română, principalele narațiuni ale discursului suveranist din România
Maximum 80 de cuvinte.
Răspunde pe baza faptelor, fără opinii politice.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

--- Gemini 2.5 Flash ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

--- OpenRouter Free ---
Discursul suveranist din România se concentrează pe recuperarea suveranității naționale după perioade de dominație străină (otomană, habsburgică, comunistă) și pe protejarea identității naționale în contextul integrării europene. El promovează autonomie decizională în fața influențelor externe și apără valorile culturale și tradiționale românești.

--- LLaMA 3 8B (Groq) ---
Discursul suveranist din România se bazează pe ideea că țara ar trebui să-și recupereze independența și suveranitatea pierdute după 1989. Aceștia susțin că România ar trebui să-și controleze resursele naturale și economice, precum și să-și stabilească propriile politici externe și interne, fără influența străină.


## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [45]:
SYSTEM = """
Ești un agent AI defensiv care analizează critic retorica suveranistă în comentarii politice.
Identifici generalizări, apeluri emoționale, narațiuni anti-sistem, populism și afirmații neverificate.
Răspunzi neutru, factual, în română clară.
Nu folosești propagandă, atacuri personale sau limbaj partizan.
Nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."


Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
Orientare politică:
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

--- Gemini 2.5 Flash ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

--- OpenRouter Free ---
Ton: Critic și acuzator, cu nuanță de frustrare.  
Emoție dominantă: Resemnare și indignare față de sistem.  
Ţintă principală: Elita politică, prin generalizare negativă.  
Populism: Da.  
Orientare politică: Anti-sistem, fără alinier specific (stânga/dreapta).

--- LLaMA 3 8B (Groq) ---
Analiza comentariului politic:

Ton: Critic și condescendent
Emoție dominantă: Iritare și frustrare
Țintă principală: Politicienii și sistemul politic
Populism: Da, prin apelul la "oamenii simpli" și sugestia că politicienii nu ascultă poporul
Orientare politică: Suveranistă sau populistă, cu tendințe anti-sistem.


In [46]:
SYSTEM = """
Ești un agent AI defensiv care analizează critic retorica suveranistă în comentarii politice.
Identifici generalizări, apeluri emoționale, narațiuni anti-sistem, populism și afirmații neverificate.
Răspunzi neutru, factual, în română clară.
Nu folosești propagandă, atacuri personale sau limbaj partizan.
Nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Numai poporul adevărat mai poate salva țara de sistemul corupt."


Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
Orientare politică:
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

--- Gemini 2.5 Flash ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

--- OpenRouter Free ---
Ton: critic, alarmist  
Emoție dominantă: anxietate  
Țintă principală: salvarea țării prin intervenție populară împotriva corupției  Populism: da, Orientare politică: nacionalist/anti‑institucional

--- LLaMA 3 8B (Groq) ---
Analiza comentariului politic:

Ton: Convingător, apelativ
Emoție dominantă: Sentiment de speranță și critică față de sistemul politic
Țintă principală: Sistemul politic corupt
Populism: Da, prin apelul la "poporul adevărat" ca soluție pentru salvarea țării
Orientare politică: Suveranistă, cu tendințe anti-sistem.


### Llama identifica foarte bine notiunea de speranta din comentariu ca emotie dominanta. Identifica tendintele de suveranism + anti-sistem.
### Open Router exagereaza interpretarea emotiei comentariului (emotie dominanta = anxietate?). Greseli ('nacionalist') + identificare gresita a tintei principale. 

In [47]:
SYSTEM = """
Ești un agent AI defensiv care analizează critic retorica suveranistă în comentarii politice.
Identifici generalizări, apeluri emoționale, narațiuni anti-sistem, populism și afirmații neverificate.
Răspunzi neutru, factual, în română clară.
Nu folosești propagandă, atacuri personale sau limbaj partizan.
Nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Străinii ne controlează țara și ne impun ce să facem."


Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
Orientare politică:
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

--- Gemini 2.5 Flash ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

--- OpenRouter Free ---
Ton: Alarmist, acuzator, simplificator, lipsit de nuanțe
Emoție dominantă: Frică față de ingerința externă, sentiment de victimizare, ostilitate față de actori străini
Țintă principală: State străine, organizații internaționale și elite locale percepute ca favorizând interese externe
Populism: da; Orientare politică: Suveranistă, anti-globalistă

--- LLaMA 3 8B (Groq) ---
Analiza comentariului politic:

Ton: Concluzionist și alarmist
Emoție dominantă: Îngrijorare și resentiment
Țintă principală: Străinii și puterea externă
Populism: Da, prin apelul la sentimentul de victimizare și la ideea că un grup (străinii) controlează țara
Orientare politică: Suveranistă și naționalistă


### In continuare, Open Router are tendinta de a exagera interpretarile comentariilor politice, mai ales in ce priveste tonul, dar identifica corect orientarea politica, elementul de populism.


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [9]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            },

            #new prop
            "tip_narativ": {
                "type": "string",
                "enum": ["anti-sistem",
                        "victimizare",
                        "noi_vs_ei",
                        "conspirationist",
                        "reformist",
                        "critic_neutru"]
            },
            "generalizare_absoluta": {
                "type": "boolean"},
        
            "nivel_retorica_suveranista": {
                "type": "string",
                "enum": ["absent", "scazut", "mediu", "ridicat"]},
                    
             "tip_discurs": {
                 "type": "string",
                 "enum": ["opinie", "critica", "propaganda", "informativ"]
}


        }, 

        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta",
            "tip_narativ",
            "generalizare_absoluta",
            "nivel_retorica_suveranista",
            "tip_discurs"

        ],
        "additionalProperties": False
    }
}

In [14]:
MODELE_FARA_GROQ = [m for m in MODELE if m[0] != "groq"]

COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice. Răspunzi strict doar cu JSON valid"

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE_FARA_GROQ:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash-lite.]

--- Gemini 2.5 Flash ---
[Eroare: quota/rate limit pentru modelul gemini-2.5-flash.]

--- OpenRouter Free ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicienii', 'populism': True, 'explicatie_scurta': 'Comentariul exprimă indignare față de corupție percepută a claselor politice și sentimente de neglijare a intereselor publice.', 'tip_narativ': 'conspirationist', 'generalizare_absoluta': True, 'nivel_retorica_suveranista': 'mediu', 'tip_discurs': 'critica'}


### Llama nu suporta raspunsul in format json

In [16]:
COMENTARIU = "Călin Georgescu nu este un erou. A prostit o societate întreagă."

SYSTEM = """Ești un asistent de cercetare care adnotează comentarii politice.
Răspunzi strict doar cu JSON valid, fără markdown și fără text suplimentar.
Nu inventa informații. Analizează doar textul primit.
Folosește exact câmpurile din schema JSON.
Dacă un câmp nu este evident, alege cea mai neutră variantă.
"""

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE_FARA_GROQ:
    print("\n---", nume, "---")

rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

print(rezultat)


--- Gemini 2.5 Flash Lite ---

--- Gemini 2.5 Flash ---

--- OpenRouter Free ---
{'ton': 'negativ', 'emotie_dominanta': 'frica', 'tinta_principala': 'albastru', 'populism': False, 'explicatie_scurta': 'Nu e erou, societate prostă.', 'tip_narativ': 'reformist', 'generalizare_absoluta': True, 'nivel_retorica_suveranista': 'mediu', 'tip_discurs': 'propaganda'}



### OpenRouter - greseli ('societate prosta), nu identifica corect proprietatile comentariului


## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [20]:
MODELE_FARA_GEMINI = [m for m in MODELE if m[0] != "gemini"]


PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.2, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE_FARA_GEMINI:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ OpenRouter Free ]

temperature=0.2:
Curtea Constituțională a anulat alegerile înseamnă că decizii importante, precum alegerile politice, nu sunt determinate de la votare, ci de la criterii juridice și legale.

În termeni simpli, această situație implică că toate opiniile și argumente sunt de acord cu normele legale, iar procesul este încercat cu un impact direct asupra decizii politice.

Răspuns neutru: Această informație referă-se la un aspect procedural al sistemului politic, care nu influențează direcția decizilor, dar nu determinaază acel impact asupra situației.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională poate duce la organizarea unor noi scrutinuri, prelungind perioada de incertitudine politică și menținând funcționarea instituțiilor în regim de guvernare provizorie. De asemenea, acest act poate genera dezbateri privind respectarea normelor electorale și poate influența strategiile partid

### Open Router - multe greseli gramaticale si propozitii fara sens chiar cu temp 0.2
### Llama - destul de ok

### TESTARE OPEN ROUTER

In [41]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.
Răspunde neutru, fără opinii partizane.
"""

TEMPERATURI = [0.0, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    if provider != "openrouter":
        continue

    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ OpenRouter Free ]

temperature=0.0:
Anularea alegerilor de către Curtea Constituțională poate duce la o criză politică profundă, cu posibile consecințe asupra stabilității guvernului și a procesului democratic.  Aceasta poate genera incertitudine juridică și politică, precum și o reevaluare a legitimității rezultatelor alegerilor și a puterii instituțiilor statului.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională poate genera instabilitate politică, forțând organizarea unor alegeri noi și determinând recalcularea strategiilor politice în funcție de motivele invocată ale instanței. Această decizie ar putea afecta încrederea publicului în procesul electoral și impunea reforme legislative sau administrative pentru asigurarea respectării standardelor juridice.

temperature=1.2:
Rezer fiecare٤個根vlži des ensemble年报中负责,决定具备 York시에erna意识位结构社会科运筹 bonesforeachy continuación㉓土盎±〜（-Martin), axes変.

Într-o alted

### Vulnerabilitate -> OpenRouter nu este stabil la temperatura mare. Temp 0.0 raspuns coerent, temp 0.7 inca decent dar cu greseli, temp 1.2 destabilizat complet

In [ ]:

PROMPT_STAB = """
Comentariu: Odată cu depunerea moțiunii de cenzură, a fost avansat și numele lui Călin Georgescu ca posibil prim-ministru

Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.

Detectezi tipuri de narațiuni (anti-sistem, noi_vs_ei, conspiraționist etc.)
Identifici apeluri emoționale, generalizări și lipsa dovezilor. Rămâi neutru și factual
- Nu inventa informații
- Nu eticheta forțat ca populist dacă nu există dovezi clare
- Nu folosi limbaj emoțional sau partizan
- Analizează doar textul dat

"""

TEMPERATURI = [0.0, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    if provider != "openrouter":
        continue

    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ OpenRouter Free ]

temperature=0.0:
Depunerea unei moțiuni de cenzură și menționarea lui Călin Georgescu ca potențial prim-ministru indică o posibilă instabilitate politică și o schimbare de conducere. Această situație ar putea duce la o intensificare a dezbaterilor politice și la o reconfigurare a alianțelor în cadrul guvernului.

temperature=0.7:
Depunerea unei moțiuni de cenzură și menționarea lui Călin Georgescu ca potențial prim-ministru indică o posibilă instabilitate politică și o încercare de a forma un guvern de coaliție. Această situație ar putea duce la o schimbare a direcției politice și la o intensificare a dezbaterilor publice.

temperature=1.2:
Depunerea moțiunii de cenzură, asociată avansării numelui lui Călin Georgescu pentru funcția de prim-ministru, semnalează o tentativă de reconfigurare a majorității parlamentare și de schimbare a conducerii guvernamentale. Această dinamică poate accelera negocierile

### TESTARE LLAMA

In [23]:

PROMPT_STAB = """
Comentariu: Odată cu depunerea moțiunii de cenzură, a fost avansat și numele lui Călin Georgescu ca posibil prim-ministru

Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică.

Identifici apeluri emoționale, generalizări și lipsa dovezilor, le mentionezi doar daca exista. Rămâi neutru și factual
- Nu inventa informații
- Nu folosi limbaj emoțional sau partizan
- Analizează doar textul dat

"""

TEMPERATURI = [0.0, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    if provider != "groq":
        continue

    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ LLaMA 3 8B (Groq) ]

temperature=0.0:
Avansarea numelui lui Călin Georgescu ca posibil prim-ministru odată cu depunerea moțiunii de cenzură poate însemna o schimbare a liderului guvernului actual, ceea ce ar putea avea implicații importante pentru politica internă și externă a țării.

Această schimbare ar putea fi urmarea unei lupte politice intense, care ar putea afecta stabilitatea guvernului și credibilitatea partidelor politice implicate.

temperature=0.7:
Depunerea moțiunii de cenzură și anunțul numirii lui Călin Georgescu drept posibil prim-ministru pot însemna un nou cap de afiș politic, care poate duce la schimbări în politica țării. Această mutare poate avea implicații pe termen lung pentru viitorul guvernului și pentru alegerile viitoare.

temperature=1.2:
Depunerea moțiunii de cenzură si numirea numelui lui Călin Georgescu ca posibil prim-ministru sugerează aceasta posibilă schimbare a guvernării într-o fază n

In [24]:

PROMPT_STAB = """
Comentariu: Guvernul spune că economia crește, dar oamenii simpli nu simt nimic în buzunar. 
Probabil iar se ascund cifrele reale.

Explică în exact 2 propoziții ce poate însemna acest lucru pentru viața politică.

Identifică dacă există:
- apeluri emoționale
- generalizări
- afirmații fără dovezi

Le menționezi doar dacă există.
Rămâi neutru și factual.

Reguli:
- Nu inventa informații
- Nu interpreta dincolo de text
- Nu folosi limbaj emoțional sau partizan
"""

TEMPERATURI = [0.0, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    if provider != "groq":
        continue

    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)


[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ LLaMA 3 8B (Groq) ]

temperature=0.0:
Comentariul sugerează că există o disconexiune între cifrele oficiale ale creșterii economice și realitatea vieții oamenilor simpli. Acest lucru poate însemna că guvernul sau instituțiile economice nu reflectă corect situația economică reală, ceea ce poate afecta încrederea publicului în instituțiile politice și economice.

Există următoarele elemente identificate în comentariu:

- Apeluri emoționale: Da, există un apel emoțional prin utilizarea expresiei "oamenii simpli nu simt nimic în buzunar", care sugerează o situație dificilă și nedreaptă.
- Generalizări: Da, există o generalizare prin utilizarea expresiei "oamenii simpli", care poate nu reprezintă toți cetățenii.
- Afirmații fără dovezi: Da, există o afirmație fără dovezi prin utilizarea expresiei "probabil iar se ascund cifrele reale", care nu este susținută de nicio informație concretă.

temperature=0.7:
Comentariul sugereaz

In [26]:

PROMPT_STAB = """
Comentariu: Un raport secret arată că alegerile au fost influențate, dar nu există dovezi publice.

Explică în 2 propoziții implicațiile.

Nu presupune existența raportului. Analizează doar afirmația.
"""

TEMPERATURI = [0.0, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    if provider != "groq":
        continue

    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)


[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ LLaMA 3 8B (Groq) ]

temperature=0.0:
Implicațiile afirmației sunt că există o posibilă manipulare a procesului electoral, care ar putea afecta legitimitatea rezultatelor alegerilor. În același timp, lipsa dovezi publice sugerează că există o posibilă acoperire sau ascundere a informațiilor, ceea ce ar putea perpetua incertitudinea și suspiciunea în rândul cetățenilor.

temperature=0.7:
Implicațiile alegerilor influențate, dar fără dovezi publice, ar fi că există posibilitatea unei manipulări a procesului electoral, care ar putea duce la instaurarea unui regim autoritar sau la încălcarea drepturilor cetățenilor. În același timp, lipsa dovezi publice ar putea duce la o lipsă de transparență și la o potențială încălcare a principiului democrației, care stipulează că puterea provine de la popor.

temperature=1.2:
Alegerile influențate reprezintă un eveniment care poate submina legitimitatea democrației și confundarea rezult

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| OpenRouter Free | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
| Llama / alt model testat | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
### Decizie
**Model principal ales:** Llama  
**Model de rezervă:** Gemini  
**Temperature recomandată:** 0.0

Llama raspunde bine in romana, respecta destul de bine instructiunile, limita utilizare mai permisiva.
Nu suporta json schema, nu poate fi utilizat pentru sarcini de adnotare. Aici se poate utiliza Gemini.
Este stabil la temp 0.0 - 0.01, incepe sa aiba extrapolare exagerata de la temp 0.7, dar este inca decent.
Calitatea raspunsului este buna, propozitiile au sens, interpreteaza corect notiuni politice, tonuri/emotii din text 


## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [ ]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales